Prelim data

In [59]:
# my jupyter notebook on my computer does not work so I used google drive, change these imports based off of your own computer
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [61]:

# CHANGE THESE PATHS ACCORDING TO YOUR OWN COMPUTER


# Paths to Excel files
inventory_path = '/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/BottleFillersInventory-ongoing.xlsx'
high_volume_event_spaces_path = '/content/drive/MyDrive/ProjectCS506/HighVolumeEventSpaces.xlsx'
building_and_kitchen_list_path = '/content/drive/MyDrive/ProjectCS506/BuildingandKitchenList.xlsx'

# Bottle Fillers Inventory
inventory_existing_stations = pd.read_excel(inventory_path)
print('Converted Bottle Fillers Inventory to inventory_existing_stations dataframe')

# High Volume Event Spaces
high_volume_event_spaces = pd.read_excel(high_volume_event_spaces_path)
print('Converted High Volume Event Spaces to high_volume_event_spaces dataframe')

# Convert Building and Kitchen List to separate CSV files
xls = pd.ExcelFile(building_and_kitchen_list_path)

# Building List
building_list = pd.read_excel(xls, sheet_name='Building List')
print('Converted Building List to building_list dataframe')

# Rooms with Kitchens
rooms_with_kitchens = pd.read_excel(xls, sheet_name='Rooms with Kitchens')
print('Converted Rooms with Kitchens to rooms_with_kitchens dataframe')

# Density Data
density_data = pd.read_csv('/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/updated_densitymap_hourly_cleaned.csv')
print('Converted Density Data to density_data dataframe')


Converted Bottle Fillers Inventory to inventory_existing_stations dataframe
Converted High Volume Event Spaces to high_volume_event_spaces dataframe
Converted Building List to building_list dataframe
Converted Rooms with Kitchens to rooms_with_kitchens dataframe
Converted Density Data to density_data dataframe


In [62]:
# Quick data inspection
print("Top 5 rows of all data before cleaning:")
print(inventory_existing_stations.head())
print(building_list.head())
print(rooms_with_kitchens.head())
print(high_volume_event_spaces.head())
print(density_data.head())


# Strip whitespace from column names for all dataframes
inventory_existing_stations.columns = inventory_existing_stations.columns.str.strip()
building_list.columns = building_list.columns.str.strip()
rooms_with_kitchens.columns = rooms_with_kitchens.columns.str.strip()
high_volume_event_spaces.columns = high_volume_event_spaces.columns.str.strip()
density_data.columns = density_data.columns.str.strip()


Top 5 rows of all data before cleaning:
  Address A          Address B    Floor   \
0       100         Ashford St     First   
1       120         Ashford St     First   
2       120         Ashford St    Second   
3         8  St. Mary's Street  Basement   
4         8  St. Mary's Street     First   

                                         Description Accessibility  \
0  Hallway across from vending machines, across f...        Public   
1                             Kitchenette, Room 116A    Restricted   
2                              Kitchenette, Room 222    Restricted   
3                               Hallway by bathrooms           NaN   
4                         Hallway by electrical room           NaN   

  Handicap Accessible Space@Bu room  Quantity       Date Installed  \
0                 Yes           191         1                  NaN   
1                 Yes          116A         1                  NaN   
2                 Yes           222         1                  N

# **Listing Most Relevant Columns From Original Data Sets for Determining Bottle-Filling Station Locations**

**density_data**


*   building_floor
*   building_desc
*   density_cnt
*   capacity
*   dc_ratio

**high_volume_event_spaces**
*   Address
*   Room Name
*   Floor
*   Good Candidate (manually created column with Yes, No, or Maybe for whether a space is a good candidate for a filling station based on its highlighted color in the excel file)

**rooms_with_kitchens**
*   Building Description (BLDGDESC)
*   Building Type (BLDGTYPE)
*   Floor (FLOORCODE)
*   Classroom Use Extensions (EXTCLASS)
*   Office Facilities Room Use Extensions (EXTOFFICE)
*   General Use Facilities Room Use Extensions (EXTGENERAL)

**building_list**
*   Building Description
*   Building Type
*   Residential Building Type
*   Mechanical Ventilation

**inventory_existing_stations**
*   Address A
*   Address B
*   Floor
*   Accessibility

# **Cleaning Inventory Existing Station Data**

* removing any columns we will not need for the project
* combining Address A and Address B column to get the full address for existing water filling stations
* changing the floor entries into numeric entries so that they can be used with the other data sets

In [63]:
inventory_existing_stations.shape

(220, 13)

In [64]:
inventory_existing_stations.head()

,Address A,Address B,Floor,Description,Accessibility,Handicap Accessible,Space@Bu room,Quantity,Date Installed,Type,Walk-Thru Notes,Completed,Average Density
0,100,Ashford St,First,"Hallway across from vending machines, across f...",Public,Yes,191,1,NaN,Combo Bottle Filler/Drinking Fountain,two additional drinking fountain locations in ...,2024-01-05,14
1,120,Ashford St,First,"Kitchenette, Room 116A",Restricted,Yes,116A,1,NaN,Office-Style Water Cooler,NaN,2024-01-05,15
2,120,Ashford St,Second,"Kitchenette, Room 222",Restricted,Yes,222,1,NaN,Office-Style Water Cooler,NaN,2024-01-05,11
3,8,St. Mary's Street,Basement,Hallway by bathrooms,NaN,NaN,B81,1,2023-12-01 00:00:00,Combo Bottle Filler/Drinking Fountain,NaN,NaT,5
4,8,St. Mary's Street,First,Hallway by electrical room,NaN,NaN,181,1,NaN,NaN,NaN,NaT,20


In [65]:
# removing rows that are empty
inventory_existing_stations = inventory_existing_stations.iloc[:, :-5]

In [66]:
# Removing columns not needed
inventory_existing_stations = inventory_existing_stations[['Address A', 'Address B', 'Floor', 'Accessibility']]

In [67]:
# Standardize Addresses and Make lower case to be used with other data
inventory_existing_stations['Address A'] = inventory_existing_stations['Address A'].astype(str)
inventory_existing_stations['Full Address'] = inventory_existing_stations['Address A'] + ' ' + inventory_existing_stations['Address B']
inventory_existing_stations.drop(['Address A', 'Address B'], axis=1, inplace=True)

# changing address to be lower case - same format as the other data sets
inventory_existing_stations['Full Address'] = inventory_existing_stations['Full Address'].str.lower()

In [68]:
# the Pool in fitrec is considered a floor, just making a comment so that we know this

# Standardize Floor Information
def standardize_floor_info(floor):
    # Handle NaN values
    if floor is None or pd.isna(floor):
        return np.nan
    # Convert all inputs to lower case strings to standardize processing
    floor_str = str(floor).strip().lower()

    # Map specific string values to their numeric equivalents
    floor_map = {
      'basement': 0, 'first': 1, 'second': 2, 'third': 3, 'fourth': 4, 'fifth': 5,
      'sixth': 6, 'seventh': 7, 'eighth': 8, 'ninth': 9, 'tenth': 10, 'eleventh': 11,
      'twelfth': 12, 'thirteenth': 13, 'fourteenth': 14, 'fifteenth': 15, 'sixteenth': 16,
      'seventeenth': 17, 'eighteenth': 18, 'nineteenth': 19, 'twentieth': 20,
      'lobby': 1, 'ground': 1, 'l1': 1, 'pool': -2, 'B': 0
    }

    # adding numeric string representations up to '30'
    for num in range(1, 31): floor_map[str(num)] = num

    # Check if the floor_str is directly in the map
    if floor_str in floor_map:
        return floor_map[floor_str]

    # Attempt to convert numbered floors like '23rd' to '23'
    elif floor_str.rstrip('thndstrd').isdigit():
        return int(floor_str.rstrip('thndstrd'))
    # Return the original string if no conditions are met
    # This line can be adjusted if you wish to handle unrecognized floors differently
    return floor_str



# Apply the function
inventory_existing_stations['Floor'] = inventory_existing_stations['Floor'].apply(standardize_floor_info)

In [69]:
pattern = r'\b(st|street|ave|avenue|road|rd)\b$'

inventory_existing_stations['address'] = inventory_existing_stations['Full Address'].str.replace(pattern, '', regex=True).str.strip()

inventory_existing_stations['Floor'] = inventory_existing_stations['Floor'].astype(str)

inventory_existing_stations['full_address'] = inventory_existing_stations['address'] + ' ' + inventory_existing_stations['Floor']


# dropping now un needed columns
inventory_existing_stations.drop(['Floor', 'Full Address', 'Accessibility'], axis=1, inplace=True)

# manually dropping row 1 and 2 because they are 'Restricted'

inventory_existing_stations = inventory_existing_stations.drop([1, 2], axis=0)

# printing cleaned data
print(f"Top 5 rows after cleaning the inventory of existing stations data:")
print(inventory_existing_stations)

Top 5 rows after cleaning the inventory of existing stations data:
                     address               full_address
0                100 ashford              100 ashford 1
3               8 st. mary's             8 st. mary's 0
4               8 st. mary's             8 st. mary's 1
5               8 st. mary's             8 st. mary's 3
6               8 st. mary's             8 st. mary's 9
..                       ...                        ...
215  110-112 cummington mall  110-112 cummington mall 2
216     33 harry agganis way     33 harry agganis way 1
217     33 harry agganis way     33 harry agganis way 2
218             1 university             1 university 1
219             1 university             1 university 3

[218 rows x 2 columns]


**Columns we kept from the inventory of existing stations data and why**


*   full_address - will be used because these are the list of addresses that have water filling stations currently installed
*   address - to be used with building data to find out mechanical ventillation and if maintained





# **Cleaning the rooms_with_kitchens data**


* removing any columns we will not be using
* make any floor B entries into floor 0 so that it is an integer value to be able to be used along with the other data sets
* keep a column only when it is believed to be publicly available, so whenever it fulfills any of these things (this is what I believe means it is a publicly avaialble kitchen)
**   a value of 'Residential' in the Building Type (BLDGTYPE)columns
**   an existing value (meaning it is not empty) in the Classroom Use Extensions (EXTCLASS)  columns
**   an existing value (meaning it is not empty) in the General Use Facilities Room Use Extensions (EXTGENERAL) column



In [70]:
# some columns that are originally not B or a number are being changed to NaN values because of this, might need to change this
rooms_with_kitchens['Floor (FLOORCODE)'] = rooms_with_kitchens['Floor (FLOORCODE)'].apply(standardize_floor_info)

# Step 2: Filter rows based on conditions
# A value of 'Residential' in BLDGTYPE or non-empty in EXTCLASS or EXTGENERAL
rooms_with_kitchens = rooms_with_kitchens[
    (rooms_with_kitchens['Building Type (BLDGTYPE)'] == 'Residential') |
    (rooms_with_kitchens['Classroom Use Extensions (EXTCLASS)'].notnull()) |
    (rooms_with_kitchens['General Use Facilities Room Use Extensions (EXTGENERAL)'].notnull())
]

# removing columns not needed
rooms_with_kitchens = rooms_with_kitchens[['Building Description (BLDGDESC)', 'Building Type (BLDGTYPE)', 'Floor (FLOORCODE)']]

# changing address to be lower case - same format as the other data sets
rooms_with_kitchens['Building Description (BLDGDESC)'] = rooms_with_kitchens['Building Description (BLDGDESC)'].str.lower()


# printing cleaned kitchens with rooms data
print(f"Top 5 rows after cleaning the rooms with kitchens data:")
print(rooms_with_kitchens.head())

Top 5 rows after cleaning the rooms with kitchens data:
                       Building Description (BLDGDESC)  \
229  915 commonwealth avenue                       ...   
230  925 commonwealth avenue                       ...   
231  925 commonwealth avenue                       ...   
232  925 commonwealth avenue                       ...   
235  925 commonwealth avenue                       ...   

                              Building Type (BLDGTYPE) Floor (FLOORCODE)  
229  Student Support                               ...                 1  
230  Athletic                                      ...                c2  
231  Athletic                                      ...                g1  
232  Athletic                                      ...                g1  
235  Athletic                                      ...                 1  


**Columns we kept from the rooms with kitchens data and why**

*   Building Description (BLDGDESC) - contains the address for the building, going to be used with floor to know where we may not want to install new filling stations
*  Floor (FLOORCODE) - contains the floor where the publicly accessible kitchen is, so we know what floors of buildings we may not want to install new filling stations because of kitchen access
*   Building Type (BLDGTYPE) - contains what type of building it is, if a building is academic or athletic, we may still want to install filling stations there even if there is a kitchen

**Cleaned data only contains rows that have the data for publicly accessible kitchens, filtered out the non publicly accessible kitchens because they are not of importance to our project and just crowd the data.**


# **Cleaning the building list data**

* removing any columns that will not be needed for the project

In [71]:
# Removing not needed columns
building_list = building_list[['Building Description', 'Building Type', 'Residential Building Type', 'Mechanical Ventilation', 'Maintained by BU FMO']]

# changing address to be lower case - same format as the other data sets
building_list['Building Description'] = building_list['Building Description'].str.lower()


pattern = r'\b(st|street|ave|avenue|road|rd)\b'
building_list['address'] = building_list['Building Description'].str.replace(pattern, '', regex=True).str.strip()


building_list.drop(['Building Description'], axis=1, inplace=True)

print(f"Top 5 rows after cleaning the building list data:")
print(building_list.head())

Top 5 rows after cleaning the building list data:
    Building Type Residential Building Type Mechanical Ventilation  \
0     Residential     Small Dormitory-Style                     No   
1        Academic                       NaN                    Yes   
2     Residential           Apartment Style                    Yes   
3     Residential                     House                Partial   
4  Administrative                       NaN                    Yes   

  Maintained by BU FMO          address  
0                  YES        1 buswell  
1                  YES     1 university  
2                  YES         10 buick  
3                  YES  10 cottage farm  
4                  YES         10 lenox  


<ipython-input-71-2c96adeff3d9>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  building_list['Building Description'] = building_list['Building Description'].str.lower()
<ipython-input-71-2c96adeff3d9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  building_list['address'] = building_list['Building Description'].str.replace(pattern, '', regex=True).str.strip()
<ipython-input-71-2c96adeff3d9>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in

**Columns we kept from the building list data and why**

*   Building Description - contains the buildings address so that this data can be used in coordination with all of the other data sets
*   Building Type - may want to prioritize installing filling stations in athletic and academic buildings over residential and administrative buildings
*  Residential Building Type - if a residential building is apartment style we will check with the kitchen data and if there is a kitchen there is no need to install a water filling station
*  Mechanical Ventilation - a.k.a. Air Conditioning, we will want to prioritize installation of new filling stations in buildings that do not have mechanical ventillation (A/C)
*  Maintained by BU FMO - if a building has maintenance, we should maybe give it higher priority because we know that installing a filling station there would be able to run properly and be fixed when broken vs not having maintenance might be a problem if we suggest to install one and then it breaks and is not fixed for long periods of time



# **Cleaning the high volume event spaces data**


We manually cleaned this data because it only contains 29 rows, in manual cleaning:
*   changed the Floor column to contain an integer entry for floor instead of a string entry for floor
*   added a new column called "Good Candidate", because on the original excel sheet, columns were highlighted to indicate their candidacy, so we created this new column that contains "Yes" if its a good candidate, "Maybe" if it is maybe a good candidate, or "No" if it is not a good candidate

In the code below:
* removing any columns that will not be needed for the project


In [72]:
# Removing not needed columns
high_volume_event_spaces = high_volume_event_spaces[['Address', 'Room Name', 'Floor', 'Good Candidate']]

# changing address to be lower case - same format as the other data sets
high_volume_event_spaces['Address'] = high_volume_event_spaces['Address'].str.lower()

print(f"Top 5 rows after cleaning the high volume event spaces data:")
print(high_volume_event_spaces.head())

Top 5 rows after cleaning the high volume event spaces data:
                        Address                                Room Name  \
0       925 commonwealth avenue           Agganis Arena Burke Club Room    
1       925 commonwealth avenue  Agganis Arena Friends of Hockey Lounge    
2            285 babcock street              Case Center Eilberg Lounge    
3   675-725 commonwealth avenue   College of Arts & Sciences Classrooms    
4        640 commowealth avenue         College of Communication Lounge    

  Floor Good Candidate  
0     1          Maybe  
1     1          Maybe  
2     3             No  
3   NaN             No  
4     1          Maybe  


<ipython-input-72-986820513e30>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  high_volume_event_spaces['Address'] = high_volume_event_spaces['Address'].str.lower()


**Columns we kept from the high volume event spaces data and why**


*   Address - with the floor and the address we can compare this data set in coordination with all of the other data sets
*   Floor - with the floor and the address we can compare this data set in coordination with all of the other data sets
*   Room Name - for the final outcome of the project, we will go back to this to find the names of the high volume event spaces if new water filling stations should be installed
*  Good Candidate - Tells us whether a space is a good candidate or not for new filling stations, prioritize good candidate high volume event spaces for installing new filling locations

# **Cleaning the density data**

* removing any columns that will not be needed for the project

In [73]:
density_data.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
0,2021-03-31,1,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,1,8,0.13
1,2021-03-31,1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,1,12,0.08
2,2021-03-31,1,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,1,6,0.17
3,2021-03-31,1,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,1,9,0.11
4,2021-03-31,1,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,1,5,0.20


In [74]:
# Remove columns not needed
density_data = density_data[['date', 'hour', 'building_type', 'building_floor', 'building_desc', 'density_cnt', 'capacity', 'dc_ratio', 'longitude', 'latitude']]

# changing address to be lower case - same format as the other data sets
density_data['building_desc'] = density_data['building_desc'].str.lower()

# remove rows where the 'hour' is 1, 2, 3, 4, or 5
density_data = density_data[~density_data['hour'].isin([1, 2, 3, 4, 5])]

pattern = r'\b(st|street|ave|avenue|road|rd)\b$'
density_data['address'] = density_data['building_desc'].str.replace(pattern, '', regex=True).str.strip()

density_data['building_floor'] = density_data['building_floor'].astype(str)

density_data['full_address'] = density_data['address'] + ' ' + density_data['building_floor']


print(f"Top 5 rows after cleaning the density data:")
print(density_data.head())

Top 5 rows after cleaning the density data:
            date  hour   building_type building_floor  \
1608  2021-03-31     6  Administrative              3   
1609  2021-03-31     6        Academic              2   
1610  2021-03-31     6        Academic              3   
1611  2021-03-31     6  Administrative              1   
1612  2021-03-31     6     Residential              2   

                building_desc  density_cnt  capacity  dc_ratio  longitude  \
1608  881 commonwealth avenue            2        70      0.03   -71.1153   
1609  808 commonwealth avenue            3       317      0.01   -71.1117   
1610  808 commonwealth avenue            1       185      0.01   -71.1117   
1611     163-179 amory street            8       103      0.08   -71.1136   
1612       112 bay state road            1         3      0.33   -71.0984   

      latitude           address        full_address  
1608   42.3514  881 commonwealth  881 commonwealth 3  
1609   42.3501  808 commonwealth  808 co

**Columns we kept from the density data and why**

*  date - going to be used with time to analyze foot traffic over time to identify the areas with high foor traffic over longer time periods
*  time - going to be used with date to analyze foot traffic over time to identify the areas with high foor traffic over longer time periods
*  density_cnt - the actual count of people detected
*  capacity - there are potential maximum capacities for spaces, which may impact the density count, making it lower in spaces with low capacities, so this needs to be factored in to our project
*  dc_ratio - may be needed to measure density for spaces with capacities
*  address - to be used with building description data to find out if a building is maintained and if a building has AC
*  full_address - to be used with inventory existing stations data to find out if existing station on that address of that building

# **Saving all cleaned data to new CSV's**

In [75]:
density_data.to_csv('/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/density_data_cleaned.csv', index=False)
inventory_existing_stations.to_csv('/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/inventory_existing_stations_cleaned.csv', index=False)
high_volume_event_spaces.to_csv('/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/high_volume_event_spaces_cleaned.csv', index=False)
building_list.to_csv('/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/building_list_cleaned.csv', index=False)
rooms_with_kitchens.to_csv('/content/drive/MyDrive/ProjectCS506/BU Sustainability: Water Bottle stations/Team B/data/rooms_with_kitchens_cleaned.csv', index=False)


print("all cleaned CSV files saved")

all cleaned CSV files saved
